# Binary Search Trees (BST)

A comprehensive guide to Binary Search Trees: definition, properties, operations, and implementation.

**Reference:** Original implementation from `origin/Algorithms_HW5.ipynb`

---

[< Previous: Dynamic Arrays](../04_Linear_Data_Structures/03_dynamic_arrays.ipynb) | [Tier 4: Algorithms & Data Structures Overview](../README.md) | [Next: AVL Trees >](02_avl_trees.ipynb)

---

## Table of Contents

1. [BST Definition & Properties](#1-bst-definition--properties)
2. [Node and Tree Implementation](#2-node-and-tree-implementation)
3. [Insert Operation](#3-insert-operation)
4. [Search Operation](#4-search-operation)
5. [Min/Max Operations](#5-minmax-operations)
6. [Successor/Predecessor](#6-successorpredecessor)
7. [Tree Traversals](#7-tree-traversals)
8. [Delete Operation](#8-delete-operation)
9. [Complexity Analysis](#9-complexity-analysis)
10. [Complete Examples](#10-complete-examples)

---

## 1. BST Definition & Properties

### What is a Binary Search Tree?

A **Binary Search Tree (BST)** is a binary tree data structure where each node satisfies the **BST property**:

> For every node `N`:
> - All values in the **left subtree** are **less than** `N.value`
> - All values in the **right subtree** are **greater than** `N.value`

### Visual Representation

```
        8
       / \
      3   10
     / \    \
    1   6    14
       / \   /
      4   7 13

BST Property Verification:
├── Node 8:  left subtree {1,3,4,6,7} < 8 < {10,13,14} right subtree  ✓
├── Node 3:  left subtree {1} < 3 < {4,6,7} right subtree             ✓
├── Node 10: left subtree {} < 10 < {13,14} right subtree             ✓
├── Node 6:  left subtree {4} < 6 < {7} right subtree                 ✓
├── Node 14: left subtree {13} < 14 < {} right subtree                ✓
└── Leaf nodes (1, 4, 7, 13): No children to verify                   ✓
```

### Why Use a BST?

The BST property enables **efficient searching**:

1. **Divide and Conquer**: At each node, we eliminate half of the remaining nodes
2. **Ordered Structure**: Inorder traversal yields sorted sequence
3. **Dynamic Operations**: Supports insert/delete while maintaining order

### Key Properties

| Property | Description |
|----------|-------------|
| Ordering | left < node < right |
| Uniqueness | Typically no duplicate keys |
| Inorder Traversal | Produces sorted sequence |
| Height | Best: O(log n), Worst: O(n) |

---

## 2. Node and Tree Implementation

### Node Class

Each node stores:
- `value`: The key/data
- `left`: Reference to left child
- `right`: Reference to right child
- `parent`: Reference to parent node (useful for deletion)

In [ ]:
from typing import Optional, Any, List, Generator
from collections import deque


class Node:
    """A node in a Binary Search Tree.
    
    Attributes:
        value: The key stored in this node.
        left: Reference to the left child (smaller values).
        right: Reference to the right child (larger values).
        parent: Reference to the parent node.
    """
    
    def __init__(self, value: Any) -> None:
        """Initialize a new node with the given value.
        
        Args:
            value: The key to store in this node.
        """
        self.value = value
        self.left: Optional['Node'] = None
        self.right: Optional['Node'] = None
        self.parent: Optional['Node'] = None
    
    def __repr__(self) -> str:
        """Return string representation of the node."""
        return f"Node({self.value})"
    
    def is_leaf(self) -> bool:
        """Check if node is a leaf (no children)."""
        return self.left is None and self.right is None
    
    def has_one_child(self) -> bool:
        """Check if node has exactly one child."""
        return (self.left is None) != (self.right is None)
    
    def has_two_children(self) -> bool:
        """Check if node has two children."""
        return self.left is not None and self.right is not None

In [ ]:
class BinarySearchTree:
    """Binary Search Tree implementation with full operations.
    
    Supports: insert, search, delete, min/max, successor/predecessor,
    and all standard traversals.
    
    Attributes:
        root: The root node of the tree.
    """
    
    def __init__(self) -> None:
        """Initialize an empty BST."""
        self.root: Optional[Node] = None
    
    def is_empty(self) -> bool:
        """Check if the tree is empty."""
        return self.root is None
    
    # ==================== INSERT ====================
    
    def insert(self, value: Any) -> Node:
        """Insert a value into the BST (iterative to avoid stack overflow).

        Args:
            value: The value to insert.

        Returns:
            The newly created node.

        Note:
            Duplicate values are not inserted (returns existing node).
        """
        if self.root is None:
            self.root = Node(value)
            return self.root

        node = self.root
        while True:
            if value < node.value:
                if node.left is None:
                    node.left = Node(value)
                    node.left.parent = node
                    return node.left
                node = node.left
            elif value > node.value:
                if node.right is None:
                    node.right = Node(value)
                    node.right.parent = node
                    return node.right
                node = node.right
            else:
                return node  # Duplicate, return existing node
    
    # ==================== SEARCH ====================
    
    def search(self, value: Any) -> Optional[Node]:
        """Search for a value in the BST.
        
        Args:
            value: The value to search for.
            
        Returns:
            The node containing the value, or None if not found.
        """
        node = self.root
        while node is not None:
            if value == node.value:
                return node
            elif value < node.value:
                node = node.left
            else:
                node = node.right
        return None
    
    def contains(self, value: Any) -> bool:
        """Check if value exists in the BST."""
        return self.search(value) is not None
    
    # ==================== MIN / MAX ====================
    
    def find_min(self, node: Optional[Node] = None) -> Optional[Node]:
        """Find the node with minimum value.
        
        Args:
            node: Starting node (defaults to root).
            
        Returns:
            Node with minimum value, or None if tree is empty.
        """
        if node is None:
            node = self.root
        if node is None:
            return None
        while node.left is not None:
            node = node.left
        return node
    
    def find_max(self, node: Optional[Node] = None) -> Optional[Node]:
        """Find the node with maximum value.
        
        Args:
            node: Starting node (defaults to root).
            
        Returns:
            Node with maximum value, or None if tree is empty.
        """
        if node is None:
            node = self.root
        if node is None:
            return None
        while node.right is not None:
            node = node.right
        return node
    
    # ==================== SUCCESSOR / PREDECESSOR ====================
    
    def successor(self, node: Node) -> Optional[Node]:
        """Find the inorder successor of a node.
        
        The successor is the node with the smallest value greater than node.value.
        
        Args:
            node: The node to find successor for.
            
        Returns:
            The successor node, or None if node has the maximum value.
        """
        # Case 1: Node has right subtree -> min of right subtree
        if node.right is not None:
            return self.find_min(node.right)
        
        # Case 2: Go up until we find a node that is a left child
        parent = node.parent
        while parent is not None and node == parent.right:
            node = parent
            parent = parent.parent
        return parent
    
    def predecessor(self, node: Node) -> Optional[Node]:
        """Find the inorder predecessor of a node.
        
        The predecessor is the node with the largest value smaller than node.value.
        
        Args:
            node: The node to find predecessor for.
            
        Returns:
            The predecessor node, or None if node has the minimum value.
        """
        # Case 1: Node has left subtree -> max of left subtree
        if node.left is not None:
            return self.find_max(node.left)
        
        # Case 2: Go up until we find a node that is a right child
        parent = node.parent
        while parent is not None and node == parent.left:
            node = parent
            parent = parent.parent
        return parent
    
    # ==================== TRAVERSALS ====================
    
    def inorder(self, node: Optional[Node] = None) -> List[Any]:
        """Inorder traversal (Left-Node-Right) - produces sorted output."""
        if node is None:
            node = self.root
        result = []
        self._inorder_recursive(node, result)
        return result
    
    def _inorder_recursive(self, node: Optional[Node], result: List[Any]) -> None:
        if node is not None:
            self._inorder_recursive(node.left, result)
            result.append(node.value)
            self._inorder_recursive(node.right, result)
    
    def preorder(self, node: Optional[Node] = None) -> List[Any]:
        """Preorder traversal (Node-Left-Right)."""
        if node is None:
            node = self.root
        result = []
        self._preorder_recursive(node, result)
        return result
    
    def _preorder_recursive(self, node: Optional[Node], result: List[Any]) -> None:
        if node is not None:
            result.append(node.value)
            self._preorder_recursive(node.left, result)
            self._preorder_recursive(node.right, result)
    
    def postorder(self, node: Optional[Node] = None) -> List[Any]:
        """Postorder traversal (Left-Right-Node)."""
        if node is None:
            node = self.root
        result = []
        self._postorder_recursive(node, result)
        return result
    
    def _postorder_recursive(self, node: Optional[Node], result: List[Any]) -> None:
        if node is not None:
            self._postorder_recursive(node.left, result)
            self._postorder_recursive(node.right, result)
            result.append(node.value)
    
    def level_order(self) -> List[Any]:
        """Level-order traversal (BFS) - visits nodes level by level."""
        if self.root is None:
            return []
        result = []
        queue = deque([self.root])
        while queue:
            node = queue.popleft()
            result.append(node.value)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        return result
    
    # ==================== DELETE ====================
    
    def delete(self, value: Any) -> bool:
        """Delete a value from the BST.
        
        Handles three cases:
        1. Leaf node: Simply remove
        2. One child: Replace with child
        3. Two children: Replace with inorder successor
        
        Args:
            value: The value to delete.
            
        Returns:
            True if deleted, False if value not found.
        """
        node = self.search(value)
        if node is None:
            return False
        self._delete_node(node)
        return True
    
    def _delete_node(self, node: Node) -> None:
        """Internal method to delete a specific node."""
        # Case 1: Leaf node (no children)
        if node.is_leaf():
            self._replace_node(node, None)
        
        # Case 2: One child
        elif node.has_one_child():
            child = node.left if node.left else node.right
            self._replace_node(node, child)
        
        # Case 3: Two children
        else:
            # Find inorder successor (minimum in right subtree)
            successor = self.find_min(node.right)
            # Copy successor's value to current node
            node.value = successor.value
            # Delete the successor (which has at most one child)
            self._delete_node(successor)
    
    def _replace_node(self, node: Node, new_node: Optional[Node]) -> None:
        """Replace a node with another node (or None)."""
        if node.parent is None:
            # Node is root
            self.root = new_node
        elif node == node.parent.left:
            node.parent.left = new_node
        else:
            node.parent.right = new_node
        
        if new_node is not None:
            new_node.parent = node.parent
    
    # ==================== UTILITY ====================
    
    def height(self, node: Optional[Node] = None) -> int:
        """Calculate the height of the tree (or subtree rooted at node).

        Uses an iterative BFS to avoid hitting Python's recursion limit on
        degenerate (sorted-input) trees.

        Returns -1 for an empty tree, 0 for a single node.
        """
        from collections import deque as _deque
        if node is None:
            node = self.root
        if node is None:
            return -1
        # BFS level-by-level; height = number of levels - 1
        queue = _deque([node])
        h = -1
        while queue:
            h += 1
            for _ in range(len(queue)):
                n = queue.popleft()
                if n.left:
                    queue.append(n.left)
                if n.right:
                    queue.append(n.right)
        return h

    def size(self, node: Optional[Node] = None) -> int:
        """Count the number of nodes in the tree (or subtree rooted at node).

        Uses an iterative traversal to avoid hitting Python's recursion limit.
        """
        if node is None:
            node = self.root
        if node is None:
            return 0
        count = 0
        stack = [node]
        while stack:
            n = stack.pop()
            count += 1
            if n.left:
                stack.append(n.left)
            if n.right:
                stack.append(n.right)
        return count
    
    def clear(self) -> None:
        """Remove all nodes from the tree."""
        self.root = None
    
    def __repr__(self) -> str:
        """Return string representation (inorder traversal)."""
        return f"BST({self.inorder()})"
    
    def __contains__(self, value: Any) -> bool:
        """Support 'in' operator."""
        return self.contains(value)
    
    def __len__(self) -> int:
        """Support len() function."""
        return self.size()

### Tree Visualization Helper

A utility function to print the tree structure:

In [ ]:
def print_tree(node: Optional[Node], level: int = 0, prefix: str = "Root: ") -> None:
    """Print a visual representation of the BST.
    
    Args:
        node: The node to start printing from.
        level: Current depth level (for indentation).
        prefix: Label prefix for the node.
    """
    if node is not None:
        print(" " * (level * 4) + prefix + str(node.value))
        if node.left is not None or node.right is not None:
            if node.left:
                print_tree(node.left, level + 1, "L--- ")
            else:
                print(" " * ((level + 1) * 4) + "L--- (empty)")
            if node.right:
                print_tree(node.right, level + 1, "R--- ")
            else:
                print(" " * ((level + 1) * 4) + "R--- (empty)")

---

## 3. Insert Operation

### Theory

Insertion maintains the BST property by:
1. Starting at the root
2. If value < current node, go left; if value > current node, go right
3. When reaching a null position, insert the new node there

### ASCII Art Visualization

```
Inserting 5 into the tree:

        8             8
       / \           / \
      3   10   →    3   10
     / \    \      / \    \
    1   6   14    1   6   14
       / \           / \
      4   7         4   7
                   /
                  5  ← New node

Path taken:
  8: 5 < 8  → go left
  3: 5 > 3  → go right
  6: 5 < 6  → go left
  4: 5 > 4  → go right (null) → INSERT HERE
```

### Complexity

| Case | Time Complexity |
|------|----------------|
| Average (balanced) | O(log n) |
| Worst (skewed) | O(n) |

In [ ]:
# Example: Building a BST step by step
bst = BinarySearchTree()

# Insert values one by one
values_to_insert = [8, 3, 10, 1, 6, 14, 4, 7, 13]

print("Building BST with values:", values_to_insert)
print("=" * 50)

for value in values_to_insert:
    bst.insert(value)
    print(f"\nAfter inserting {value}:")
    print_tree(bst.root)

In [ ]:
# Insert 5 and see where it goes
print("Inserting 5...")
print("Path: 8 (5<8 →L) → 3 (5>3 →R) → 6 (5<6 →L) → 4 (5>4 →R) → INSERT")
bst.insert(5)
print("\nTree after inserting 5:")
print_tree(bst.root)

---

## 4. Search Operation

### Theory

Search uses the BST property to find values efficiently:
1. Compare target with current node
2. If equal, found!
3. If target < node, search left subtree
4. If target > node, search right subtree
5. If null reached, value not in tree

### ASCII Art Visualization

```
Searching for 6:

        [8]    ← 6 < 8, go left
       /   \
     [3]    10  ← 6 > 3, go right
     / \
    1  [6]  ← Found!
       / \
      4   7

Comparisons: 8 → 3 → 6 (3 comparisons)


Searching for 9 (not in tree):

        [8]    ← 9 > 8, go right
       /   \
      3   [10]  ← 9 < 10, go left
             \
              14
              
Left child of 10 is NULL → Not found!
```

### Complexity

| Case | Time Complexity |
|------|----------------|
| Average (balanced) | O(log n) |
| Worst (skewed) | O(n) |

In [ ]:
# Search examples
print("Current tree:")
print_tree(bst.root)
print()

# Search for existing values
for value in [6, 13, 8, 1]:
    result = bst.search(value)
    print(f"Search for {value}: {'Found' if result else 'Not found'} → {result}")

print()

# Search for non-existing values
for value in [9, 2, 15]:
    result = bst.search(value)
    print(f"Search for {value}: {'Found' if result else 'Not found'}")

print()

# Using 'in' operator
print(f"6 in bst: {6 in bst}")
print(f"9 in bst: {9 in bst}")

---

## 5. Min/Max Operations

### Theory

Due to the BST property:
- **Minimum**: Always in the leftmost node (keep going left)
- **Maximum**: Always in the rightmost node (keep going right)

### ASCII Art Visualization

```
Finding Minimum:                Finding Maximum:

        8                              8
       / \                            / \
      3   10                         3   10
     / \    \                       / \    \
   [1]  6   14                     1   6   [14]
    ↑                                        ↑
    Leftmost = MIN                  Rightmost = MAX

Path to MIN: 8 → 3 → 1 (stop, no left child)
Path to MAX: 8 → 10 → 14 (stop, no right child)
```

### Complexity

| Case | Time Complexity |
|------|----------------|
| Average (balanced) | O(log n) |
| Worst (skewed) | O(n) |

In [ ]:
# Min/Max examples
print("Current tree:")
print_tree(bst.root)
print()

min_node = bst.find_min()
max_node = bst.find_max()

print(f"Minimum value: {min_node.value if min_node else 'Tree is empty'}")
print(f"Maximum value: {max_node.value if max_node else 'Tree is empty'}")
print()

# Min/Max in subtrees
node_3 = bst.search(3)
print(f"Minimum in subtree rooted at 3: {bst.find_min(node_3).value}")
print(f"Maximum in subtree rooted at 3: {bst.find_max(node_3).value}")

---

## 6. Successor/Predecessor

### Theory

**Inorder Successor**: Next node in inorder traversal (smallest value > current)
- If right subtree exists: minimum of right subtree
- Otherwise: first ancestor where node is in left subtree

**Inorder Predecessor**: Previous node in inorder traversal (largest value < current)
- If left subtree exists: maximum of left subtree
- Otherwise: first ancestor where node is in right subtree

### ASCII Art Visualization

```
Inorder: 1, 3, 4, 5, 6, 7, 8, 10, 13, 14

        8
       / \
      3   10
     / \    \
    1   6   14
       / \   /
      4   7 13
       \
        5

Successor of 6:
  - 6 has right subtree → find min of right subtree
  - Min of {7} = 7
  - Successor(6) = 7

Successor of 7:
  - 7 has no right subtree
  - Go up: 7→6 (7 is right child, continue)
  - Go up: 6→3 (6 is right child, continue)
  - Go up: 3→8 (3 is left child, STOP)
  - Successor(7) = 8

Predecessor of 8:
  - 8 has left subtree → find max of left subtree
  - Max of {1,3,4,5,6,7} = 7
  - Predecessor(8) = 7
```

### Complexity

| Case | Time Complexity |
|------|----------------|
| Average | O(log n) |
| Worst | O(n) |

In [ ]:
# Successor/Predecessor examples
print("Current tree:")
print_tree(bst.root)
print()

print("Inorder traversal:", bst.inorder())
print()

# Test successor for various nodes
print("Successors:")
for value in [1, 3, 6, 7, 8, 13, 14]:
    node = bst.search(value)
    succ = bst.successor(node)
    print(f"  Successor({value}) = {succ.value if succ else 'None (max element)'}")

print()

# Test predecessor for various nodes
print("Predecessors:")
for value in [1, 3, 6, 7, 8, 13, 14]:
    node = bst.search(value)
    pred = bst.predecessor(node)
    print(f"  Predecessor({value}) = {pred.value if pred else 'None (min element)'}")

---

## 7. Tree Traversals

### Theory

Four standard ways to visit all nodes:

| Traversal | Order | Use Case |
|-----------|-------|----------|
| **Inorder** | Left → Node → Right | Sorted output |
| **Preorder** | Node → Left → Right | Copy tree, prefix expression |
| **Postorder** | Left → Right → Node | Delete tree, postfix expression |
| **Level-order** | Level by level (BFS) | Print by depth |

### ASCII Art Visualization

```
        8
       / \
      3   10
     / \    \
    1   6   14

Inorder (L-N-R):   1 → 3 → 6 → 8 → 10 → 14   ← SORTED!
                   ↑   ↑   ↑   ↑    ↑    ↑
                   L   N   R   N    L    R

Preorder (N-L-R):  8 → 3 → 1 → 6 → 10 → 14
                   ↑   ↑   ↑   ↑    ↑    ↑
                   N   L   L   R    R    R

Postorder (L-R-N): 1 → 6 → 3 → 14 → 10 → 8
                   ↑   ↑   ↑    ↑    ↑   ↑
                   L   R   N    R    N   N

Level-order:       8 → 3 → 10 → 1 → 6 → 14
                   |   |____|   |___|___|
                Level 0  Level 1  Level 2
```

### Complexity

| Traversal | Time | Space |
|-----------|------|-------|
| All | O(n) | O(h) where h = height |

In [ ]:
# Create a fresh tree for traversal demonstration
demo_tree = BinarySearchTree()
for value in [8, 3, 10, 1, 6, 14]:
    demo_tree.insert(value)

print("Tree structure:")
print_tree(demo_tree.root)
print()

print("=" * 50)
print("TRAVERSALS")
print("=" * 50)

print(f"\nInorder (L-N-R):   {demo_tree.inorder()}")
print("  → Produces SORTED output!")

print(f"\nPreorder (N-L-R):  {demo_tree.preorder()}")
print("  → Root first, then subtrees")

print(f"\nPostorder (L-R-N): {demo_tree.postorder()}")
print("  → Children before parent")

print(f"\nLevel-order (BFS): {demo_tree.level_order()}")
print("  → Level by level, left to right")

In [ ]:
# Visual trace of inorder traversal
print("Inorder Traversal Step-by-Step:")
print("="*40)
print("""
        8
       / \\
      3   10
     / \\    \\
    1   6   14

Stack trace (recursive calls):
  inorder(8)
    ├─ inorder(3)
    │    ├─ inorder(1)
    │    │    ├─ inorder(None) → return
    │    │    ├─ VISIT 1 ✓
    │    │    └─ inorder(None) → return
    │    ├─ VISIT 3 ✓
    │    └─ inorder(6)
    │         ├─ inorder(None) → return
    │         ├─ VISIT 6 ✓
    │         └─ inorder(None) → return
    ├─ VISIT 8 ✓
    └─ inorder(10)
         ├─ inorder(None) → return
         ├─ VISIT 10 ✓
         └─ inorder(14)
              ├─ inorder(None) → return
              ├─ VISIT 14 ✓
              └─ inorder(None) → return

Result: [1, 3, 6, 8, 10, 14] ← Sorted!
""")

---

## 8. Delete Operation

### Theory

Deletion is the most complex BST operation. Three cases:

| Case | Description | Action |
|------|-------------|--------|
| **Leaf** | No children | Simply remove |
| **One child** | Exactly one child | Replace with child |
| **Two children** | Both children exist | Replace with inorder successor |

### ASCII Art Visualization

#### Case 1: Delete Leaf Node (delete 4)

```
        8               8
       / \             / \
      3   10    →     3   10
     / \    \        / \    \
    1   6   14      1   6   14
       /               
     [4] ← delete      (removed)

Action: Simply remove node 4
        Parent's (6) left pointer → NULL
```

#### Case 2: Delete Node with One Child (delete 10)

```
        8                8
       / \              / \
      3  [10]   →      3   14
     / \    \         / \   /
    1   6   14       1   6 13
           /
          13

Action: Replace 10 with its only child (14)
        14 takes 10's position
        Parent (8) → 14
```

#### Case 3: Delete Node with Two Children (delete 3)

```
Step 1: Find inorder successor of 3
        Successor = min of right subtree = 4

        8               
       / \              
     [3]  10            
     / \    \           
    1   6   14          
       /   /
      4   13
       \
        5
      
Step 2: Copy successor's value to node being deleted
        Node 3's value becomes 4

Step 3: Delete the original successor (4)
        4 has one child (5), use Case 2

        8               
       / \              
      4   10     ← 3 replaced by 4       
     / \    \           
    1   6   14          
       /   /
      5   13     ← 4 replaced by 5
```

### Complexity

| Case | Time Complexity |
|------|----------------|
| Average (balanced) | O(log n) |
| Worst (skewed) | O(n) |

In [ ]:
# Create tree for deletion examples
def create_example_tree() -> BinarySearchTree:
    """Create a BST for deletion examples."""
    tree = BinarySearchTree()
    for value in [8, 3, 10, 1, 6, 14, 4, 7, 13, 5]:
        tree.insert(value)
    return tree

print("Original tree:")
tree1 = create_example_tree()
print_tree(tree1.root)
print(f"\nInorder: {tree1.inorder()}")

In [ ]:
# Case 1: Delete leaf node (5)
print("=" * 50)
print("CASE 1: Delete Leaf Node (5)")
print("=" * 50)

tree1 = create_example_tree()
print("Before deletion:")
print_tree(tree1.root)

tree1.delete(5)

print("\nAfter deleting 5 (leaf):")
print_tree(tree1.root)
print(f"\nInorder: {tree1.inorder()}")

In [ ]:
# Case 2: Delete node with one child (10 has child 14)
print("=" * 50)
print("CASE 2: Delete Node with One Child (10)")
print("=" * 50)

tree2 = create_example_tree()
print("Before deletion:")
print_tree(tree2.root)

tree2.delete(10)

print("\nAfter deleting 10 (has one child: 14):")
print_tree(tree2.root)
print(f"\nInorder: {tree2.inorder()}")

In [ ]:
# Case 3: Delete node with two children (3 has children 1 and 6)
print("=" * 50)
print("CASE 3: Delete Node with Two Children (3)")
print("=" * 50)

tree3 = create_example_tree()
print("Before deletion:")
print_tree(tree3.root)

# Find successor of 3 (it's 4, the minimum in right subtree)
node_3 = tree3.search(3)
succ = tree3.successor(node_3)
print(f"\nInorder successor of 3: {succ.value}")
print("Strategy: Replace 3 with 4, then delete original 4")

tree3.delete(3)

print("\nAfter deleting 3:")
print_tree(tree3.root)
print(f"\nInorder: {tree3.inorder()}")

In [ ]:
# Delete root node
print("=" * 50)
print("SPECIAL CASE: Delete Root Node (8)")
print("=" * 50)

tree4 = create_example_tree()
print("Before deletion:")
print_tree(tree4.root)

tree4.delete(8)

print("\nAfter deleting root (8):")
print_tree(tree4.root)
print(f"\nInorder: {tree4.inorder()}")

---

## 9. Complexity Analysis

### Time Complexity Summary

| Operation | Average Case | Worst Case | Notes |
|-----------|-------------|------------|-------|
| **Search** | O(log n) | O(n) | Binary search through tree |
| **Insert** | O(log n) | O(n) | Find position + create node |
| **Delete** | O(log n) | O(n) | Find + restructure |
| **Min/Max** | O(log n) | O(n) | Follow left/right path |
| **Successor/Predecessor** | O(log n) | O(n) | Subtree or ancestor search |
| **Traversals** | O(n) | O(n) | Must visit all nodes |

### Space Complexity

| Operation | Space |
|-----------|-------|
| Tree storage | O(n) |
| Recursive traversal | O(h) stack space |
| Level-order traversal | O(w) where w = max width |

### Worst Case: Skewed Tree

```
Balanced Tree (h = log n):        Skewed Tree (h = n):

        4                         1
       / \                         \
      2   6                         2
     / \ / \                         \
    1  3 5  7                         3
                                       \
Height = 2 = log₂(7)                    4
Search 7: 3 comparisons                  \
                                          5
                                           \
                                            6
                                             \
                                              7
                                   Height = 6 = n-1
                                   Search 7: 7 comparisons
```

In [ ]:
# Demonstrate worst case: inserting sorted data
print("WORST CASE: Inserting Sorted Data")
print("=" * 50)

# Sorted insertion creates a skewed tree (linked list)
skewed_tree = BinarySearchTree()
sorted_values = [1, 2, 3, 4, 5, 6, 7]

print(f"Inserting: {sorted_values}")
for v in sorted_values:
    skewed_tree.insert(v)

print("\nResulting tree (skewed right):")
print_tree(skewed_tree.root)

print(f"\nNumber of nodes: {skewed_tree.size()}")
print(f"Tree height: {skewed_tree.height()}")
print(f"Optimal height (balanced): {len(sorted_values).bit_length() - 1}")
print("\nThis tree degrades to a linked list!")

In [ ]:
# Demonstrate best case: balanced insertion
print("BEST CASE: Balanced Insertion Order")
print("=" * 50)

balanced_tree = BinarySearchTree()
# Insert middle element first, then balance around it
balanced_values = [4, 2, 6, 1, 3, 5, 7]  # Level-order of balanced tree

print(f"Inserting: {balanced_values}")
for v in balanced_values:
    balanced_tree.insert(v)

print("\nResulting tree (balanced):")
print_tree(balanced_tree.root)

print(f"\nNumber of nodes: {balanced_tree.size()}")
print(f"Tree height: {balanced_tree.height()}")
print("\nThis is an optimal BST!")

In [ ]:
import time
import random

def benchmark_bst(n: int, ordered: bool = False) -> dict:
    """Benchmark BST operations.
    
    Args:
        n: Number of elements
        ordered: If True, insert in sorted order (worst case)
        
    Returns:
        Dict with timing results
    """
    tree = BinarySearchTree()
    
    if ordered:
        values = list(range(n))
    else:
        values = list(range(n))
        random.shuffle(values)
    
    # Measure insert time
    start = time.perf_counter()
    for v in values:
        tree.insert(v)
    insert_time = time.perf_counter() - start
    
    # Measure search time
    search_values = random.sample(values, min(100, n))
    start = time.perf_counter()
    for v in search_values:
        tree.search(v)
    search_time = time.perf_counter() - start
    
    return {
        'n': n,
        'ordered': ordered,
        'height': tree.height(),
        'insert_time': insert_time,
        'search_time': search_time / len(search_values) * 1000  # ms per search
    }

print("Performance Comparison: Random vs Sorted Insertion")
print("=" * 60)
print(f"{'n':>8} {'Type':>10} {'Height':>8} {'Insert(s)':>12} {'Search(ms)':>12}")
print("-" * 60)

for n in [100, 500, 1000]:
    # Random insertion
    result_random = benchmark_bst(n, ordered=False)
    print(f"{result_random['n']:>8} {'Random':>10} {result_random['height']:>8} "
          f"{result_random['insert_time']:>12.6f} {result_random['search_time']:>12.6f}")
    
    # Sorted insertion (worst case)
    result_sorted = benchmark_bst(n, ordered=True)
    print(f"{result_sorted['n']:>8} {'Sorted':>10} {result_sorted['height']:>8} "
          f"{result_sorted['insert_time']:>12.6f} {result_sorted['search_time']:>12.6f}")
    print()

---

## 10. Complete Examples

### Example 1: Building and Querying a BST

In [ ]:
# Complete example: Build, query, and modify a BST
print("="*60)
print("COMPLETE EXAMPLE: Student Grade Management")
print("="*60)

# Create a BST to store student IDs
students = BinarySearchTree()

# Insert student IDs
student_ids = [50, 30, 70, 20, 40, 60, 80, 35, 65]
print(f"Enrolling students with IDs: {student_ids}")
for sid in student_ids:
    students.insert(sid)

print("\nStudent roster (tree structure):")
print_tree(students.root)

print(f"\nTotal students: {len(students)}")
print(f"Student IDs (sorted): {students.inorder()}")

# Search for a student
search_id = 40
print(f"\nSearching for student {search_id}: {'Found' if search_id in students else 'Not found'}")

# Find extreme IDs
print(f"\nLowest ID: {students.find_min().value}")
print(f"Highest ID: {students.find_max().value}")

# Graduate some students (delete)
graduating = [20, 50, 70]
print(f"\nGraduating students: {graduating}")
for sid in graduating:
    students.delete(sid)

print("\nUpdated roster:")
print_tree(students.root)
print(f"Remaining students: {students.inorder()}")

### Example 2: All Deletion Cases

In [ ]:
# Comprehensive deletion demonstration
print("="*60)
print("DELETION CASES DEMONSTRATION")
print("="*60)

def demo_deletion(values: list, delete_value: int, case_name: str) -> None:
    """Demonstrate a deletion case."""
    tree = BinarySearchTree()
    for v in values:
        tree.insert(v)
    
    print(f"\n{case_name}")
    print("-" * 40)
    print("Before:")
    print_tree(tree.root)
    print(f"Inorder: {tree.inorder()}")
    
    # Show node details
    node = tree.search(delete_value)
    if node:
        left = node.left.value if node.left else "None"
        right = node.right.value if node.right else "None"
        print(f"\nDeleting {delete_value} (left={left}, right={right})")
    
    tree.delete(delete_value)
    
    print("\nAfter:")
    print_tree(tree.root)
    print(f"Inorder: {tree.inorder()}")

# Case 1: Leaf
demo_deletion([50, 30, 70, 20, 40], 20, "CASE 1: Delete Leaf (20)")

# Case 2a: One child (left)
demo_deletion([50, 30, 70, 20], 30, "CASE 2a: One Child - Left (30)")

# Case 2b: One child (right)
demo_deletion([50, 30, 70, 40], 30, "CASE 2b: One Child - Right (30)")

# Case 3: Two children
demo_deletion([50, 30, 70, 20, 40, 35], 30, "CASE 3: Two Children (30)")

### Example 3: Traversal Applications

In [ ]:
# Practical applications of traversals
print("="*60)
print("TRAVERSAL APPLICATIONS")
print("="*60)

tree = BinarySearchTree()
for v in [50, 30, 70, 20, 40, 60, 80]:
    tree.insert(v)

print("\nTree:")
print_tree(tree.root)

# Application 1: Sorting
print("\n1. SORTING with Inorder Traversal")
print(f"   Sorted sequence: {tree.inorder()}")

# Application 2: Tree copying (preorder)
print("\n2. COPY TREE Order (Preorder)")
print(f"   Insert order to copy: {tree.preorder()}")
print("   (Insert in this order creates identical tree)")

# Verify copy
copy_tree = BinarySearchTree()
for v in tree.preorder():
    copy_tree.insert(v)
print(f"   Copied tree inorder: {copy_tree.inorder()}")

# Application 3: Safe deletion order (postorder)
print("\n3. SAFE DELETE Order (Postorder)")
print(f"   Delete order: {tree.postorder()}")
print("   (Deleting in this order = children before parents)")

# Application 4: Level-by-level processing
print("\n4. LEVEL-BY-LEVEL Processing")
print(f"   BFS order: {tree.level_order()}")
print("   Level 0: [50]")
print("   Level 1: [30, 70]")
print("   Level 2: [20, 40, 60, 80]")

### Example 4: Finding k-th Smallest Element

In [ ]:
def kth_smallest(tree: BinarySearchTree, k: int) -> Optional[int]:
    """Find the k-th smallest element in the BST.
    
    Uses the property that inorder traversal gives sorted sequence.
    
    Args:
        tree: The BST to search
        k: Position (1-indexed)
        
    Returns:
        The k-th smallest value, or None if k is out of range
    """
    sorted_values = tree.inorder()
    if 1 <= k <= len(sorted_values):
        return sorted_values[k - 1]
    return None

# Example
tree = BinarySearchTree()
for v in [50, 30, 70, 20, 40, 60, 80, 35, 45]:
    tree.insert(v)

print("Tree:")
print_tree(tree.root)
print(f"\nInorder (sorted): {tree.inorder()}")
print()

for k in range(1, 10):
    result = kth_smallest(tree, k)
    print(f"  {k}-th smallest: {result}")

### Example 5: Validate BST Property

In [ ]:
def is_valid_bst(node: Optional[Node], 
                 min_val: float = float('-inf'), 
                 max_val: float = float('inf')) -> bool:
    """Check if a binary tree satisfies the BST property.
    
    Args:
        node: Current node to check
        min_val: Minimum allowed value
        max_val: Maximum allowed value
        
    Returns:
        True if valid BST, False otherwise
    """
    if node is None:
        return True
    
    # Check current node's value is within bounds
    if node.value <= min_val or node.value >= max_val:
        return False
    
    # Recursively check subtrees with updated bounds
    return (is_valid_bst(node.left, min_val, node.value) and
            is_valid_bst(node.right, node.value, max_val))

# Test with valid BST
tree = BinarySearchTree()
for v in [50, 30, 70, 20, 40]:
    tree.insert(v)

print("Valid BST:")
print_tree(tree.root)
print(f"Is valid BST: {is_valid_bst(tree.root)}")

# Create an invalid BST manually
print("\nInvalid tree (manual construction):")
invalid_root = Node(50)
invalid_root.left = Node(30)
invalid_root.right = Node(70)
invalid_root.left.right = Node(60)  # INVALID: 60 > 50 but in left subtree!

print_tree(invalid_root)
print(f"Is valid BST: {is_valid_bst(invalid_root)}")
print("(60 is in left subtree of 50 but 60 > 50 - violates BST property)")

---

## Randomized BST (Treap)

A treap (tree + heap) is a randomized BST where each node has a random priority. The tree is maintained as a BST by key and a heap by priority. This ensures O(log n) expected height without explicit rebalancing like AVL or Red-Black trees. Treaps are simpler to implement and useful when you need a balanced BST without the complexity of rotation rules.

In [ ]:
import random

class TreapNode:
    def __init__(self, key, priority=None):
        self.key = key
        self.priority = priority if priority is not None else random.random()
        self.left = None
        self.right = None

def rotate_right(node):
    """Right rotation: left child becomes new root."""
    new_root = node.left
    node.left = new_root.right
    new_root.right = node
    return new_root

def rotate_left(node):
    """Left rotation: right child becomes new root."""
    new_root = node.right
    node.right = new_root.left
    new_root.left = node
    return new_root

def treap_insert(root, key):
    """Insert key maintaining BST order and heap priority."""
    if root is None:
        return TreapNode(key)
    if key < root.key:
        root.left = treap_insert(root.left, key)
        if root.left.priority > root.priority:
            root = rotate_right(root)
    elif key > root.key:
        root.right = treap_insert(root.right, key)
        if root.right.priority > root.priority:
            root = rotate_left(root)
    return root

def treap_inorder(root):
    """In-order traversal returns sorted keys."""
    if root is None:
        return []
    return treap_inorder(root.left) + [root.key] + treap_inorder(root.right)

def treap_height(root):
    if root is None:
        return 0
    return 1 + max(treap_height(root.left), treap_height(root.right))

# Insert sorted data (worst case for regular BST)
root = None
data = list(range(1, 21))  # sorted 1..20
for val in data:
    root = treap_insert(root, val)

print(f"Inserted {len(data)} sorted values")
print(f"Treap height: {treap_height(root)} (regular BST would be {len(data)})")
print(f"In-order: {treap_inorder(root)}")


Even with sorted input (worst case for a regular BST which becomes a linked list of height 20), the treap maintains a balanced height of ~6-8 thanks to random priorities.

---

## Summary

### Key Takeaways

1. **BST Property**: `left < node < right` enables efficient O(log n) operations

2. **Operations Complexity**:
   - Average case: O(log n) for search, insert, delete
   - Worst case: O(n) when tree becomes skewed

3. **Deletion Cases**:
   - Leaf: Simply remove
   - One child: Replace with child
   - Two children: Replace with inorder successor

4. **Traversals**:
   - Inorder gives sorted sequence
   - Preorder useful for copying
   - Postorder useful for deletion
   - Level-order for breadth-first processing

5. **Limitations**:
   - No guarantee of balance
   - Sorted insertion degrades to O(n)
   - Solution: Self-balancing trees (AVL, Red-Black)

### What's Next?

- **AVL Trees**: Self-balancing BST with height difference ≤ 1
- **Red-Black Trees**: Self-balancing with color properties
- **B-Trees**: Multi-way search trees for databases

---

## Exercises

### Exercise 1: Gene Expression Range Query (1 star)

Insert a list of (gene_name, expression_value) pairs into a BST using expression values as keys. Implement a `range_query` method that returns all genes with expression values between two thresholds.

**Biological context**: In RNA-seq analysis, you often need to find all genes within a particular expression range (e.g., genes with log2 fold-change between 1.5 and 3.0) to identify moderately up-regulated genes.

In [ ]:
class GeneNode:
    def __init__(self, expression: float, gene_name: str):
        self.expression = expression
        self.gene_name = gene_name
        self.left = None
        self.right = None


class GeneExpressionBST:
    def __init__(self):
        self.root = None

    def insert(self, expression: float, gene_name: str) -> None:
        # YOUR CODE HERE
        pass

    def range_query(self, low: float, high: float) -> list[tuple[str, float]]:
        """Return all (gene_name, expression) pairs with low <= expression <= high."""
        results = []
        # YOUR CODE HERE
        return results


# Test
# gene_data = [
#     (2.3, 'BRCA1'), (0.5, 'TP53'), (4.1, 'MYC'), (1.8, 'EGFR'),
#     (3.2, 'KRAS'), (0.9, 'PTEN'), (2.7, 'CDH1'), (5.0, 'AKT1'),
# ]
# bst = GeneExpressionBST()
# for expr, gene in gene_data:
#     bst.insert(expr, gene)
# results = bst.range_query(1.5, 3.5)
# print('Genes with expression between 1.5 and 3.5:')
# for gene, expr in sorted(results, key=lambda x: x[1]):
#     print(f'  {gene}: {expr}')

### Exercise 2: BST Balance Check (2 stars)

Implement `is_balanced(root)` that returns `True` if a BST is height-balanced (the absolute height difference between left and right subtrees is at most 1 at every node). Build one BST by inserting sorted data and another by inserting shuffled data, then compare.

**Biological context**: When indexing sorted genomic coordinates (chromosome positions always arrive in order), a naive BST degenerates into a linked list with O(n) lookup — demonstrating why self-balancing trees like AVL or Red-Black are used in genome browsers.

In [ ]:
def tree_height(node) -> int:
    """Return the height of the subtree rooted at node."""
    # YOUR CODE HERE
    pass


def is_balanced(node) -> bool:
    """Return True if the BST rooted at node is height-balanced."""
    # YOUR CODE HERE
    pass


# Test with sorted vs random data
# from typing import Optional
# import random
#
# def insert_bst(root, val):
#     if root is None:
#         return Node(val)
#     if val < root.data:
#         root.left = insert_bst(root.left, val)
#     else:
#         root.right = insert_bst(root.right, val)
#     return root
#
# values = list(range(1, 16))
# sorted_root = None
# for v in values:
#     sorted_root = insert_bst(sorted_root, v)
# print(f'Sorted insert — height: {tree_height(sorted_root)}, balanced: {is_balanced(sorted_root)}')
#
# random.shuffle(values)
# random_root = None
# for v in values:
#     random_root = insert_bst(random_root, v)
# print(f'Random insert — height: {tree_height(random_root)}, balanced: {is_balanced(random_root)}')

---

[< Previous: Dynamic Arrays](../04_Linear_Data_Structures/03_dynamic_arrays.ipynb) | [Tier 4: Algorithms & Data Structures Overview](../README.md) | [Next: AVL Trees >](02_avl_trees.ipynb)

---